In [0]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
df = spark.sql('''
SELECT * 
FROM workspace.`site-speed-recommendation`.combined
WHERE performance_score is not null
''').toPandas()

df_desktop = spark.sql('''
SELECT * 
FROM workspace.`site-speed-recommendation`.combined
WHERE performance_score is not null and strategy="desktop"
''').toPandas()

df_mobile = spark.sql('''
SELECT * 
FROM workspace.`site-speed-recommendation`.combined
WHERE performance_score is not null and strategy="mobile"
''').toPandas()

In [0]:
df[df["domain"]=="https://www.wildfang.com/"]

In [0]:
df_mobile.columns

In [0]:
df_mobile['largest-contentful-paint'].quantile([0.25,0.5,0.75,0.9,0.99,0.999])

In [0]:
df_mobile[df_mobile['largest-contentful-paint'] > 135672.349547]

In [0]:
df_mobile.shape

In [0]:
dataset.columns

In [0]:
dataset.dtypes

- LCP ranges:
1. 0 - 2.5s good
2. 2.5 - 4s needs improvement
3. 4+s bad
- CLS range:
1. 0 - 0.1 good
2. 0.1 - 0.25 needs improvement
3. 0.25+ bad
- INP range:
1. 0 - 200ms good
2. 200 - 500 ms needs improvement
3. 500 + bad    

In [0]:
def score_lcp(row):
    if row['largest-contentful-paint'] <= 2500:
        return "fast"
    elif row['largest-contentful-paint'] <= 4000:
        return "moderate"
    else:
        return "slow"

def score_cls(row):
    if row['cumulative-layout-shift'] <= .1:
        return "fast"
    elif row['cumulative-layout-shift'] <= .25:
        return "moderate"
    else:
        return "slow"
    
def score_inp(row):
    if row['INTERACTION_TO_NEXT_PAINT'] <= 100:
        return "fast"
    elif row['INTERACTION_TO_NEXT_PAINT'] <= 300:
        return "moderate"
    else:
        return "slow"
    
def score_fcp(row):
    if row['first-contentful-paint'] <= 1800:
        return "fast"
    elif row['first-contentful-paint'] <= 3000:
        return "moderate"
    else:
        return "slow"
    
def score_tbt(row):
    if row['total-blocking-time'] <= 200:
        return "fast"
    elif row['total-blocking-time'] <= 600:
        return "moderate"
    else:
        return "slow"
    
def score_si(row):
    if row['speed-index'] <= 3400:
        return "fast"
    elif row['speed-index'] <= 5800:
        return "moderate"
    else:
        return "slow"

In [0]:
dataset["lcp_rank"] = dataset.apply(score_lcp, axis=1)
dataset["cls_rank"] = dataset.apply(score_cls, axis=1)
dataset["inp_rank"] = dataset.apply(score_inp, axis=1)
dataset["fcp_rank"] = dataset.apply(score_fcp, axis=1)
dataset["tbt_rank"] = dataset.apply(score_tbt, axis=1)
dataset["si_rank"]  = dataset.apply(score_si, axis=1)

In [0]:
dataset.shape

In [0]:
non_executed_rows = dataset[~dataset['performance_score'].isna()]
non_executed_rows.describe().T


In [0]:
non_executed_rows.isna().sum(axis=0)

In [0]:
def print_stats(df, col):
    fig, ax = plt.subplots(2, 1, figsize=(4,4))
    ax[0].hist(df[col], bins=20)
    ax[1] = df[col].plot(kind="box", logx=True, vert=False)

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print("lower: ", lower)
    print("upper: ", upper)
    print(df[col].describe(percentiles=[0.9,0.95,0.99,0.995,0.999]))


In [0]:
print_stats(dataset, "largest-contentful-paint")

In [0]:
print_stats(dataset, "cumulative-layout-shift")


In [0]:
sns.heatmap(non_executed_rows.drop(
    columns=["domain","url","strategy", "lcp_rank", "cls_rank", "inp_rank", "fcp_rank", "tbt_rank", "si_rank"]
).corr().abs())